# 02 — Pré-processamento Base

Este notebook realiza o pré-processamento base do dataset EEG utilizado no projeto. Essa etapa é comum a todos os modelos que serão avaliados posteriormente, como CNN, LSTM e Transformer.

Como o dataset já possui características espectrais extraídas dos sinais EEG, não são aplicadas transformadas como FFT ou PSD nesta etapa. O objetivo aqui é validar, limpar, dividir e padronizar os dados antes das transformações específicas de cada modelo.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.preprocessing.base_preprocessing import (
    BasePreprocessingConfig,
    prepare_base_data,
)

config = BasePreprocessingConfig(
    test_size=0.2,
    random_state=42,
    stratify=True,
    remove_duplicates=True,
    handle_missing=True,
    imputation_strategy="median",
    outlier_strategy="clip_iqr",
    iqr_multiplier=1.5,
    scale_data=True,
)

result = prepare_base_data(config)

result.preprocessing_report

## Relatório do pré-processamento

O relatório mostra as principais informações da execução do pipeline. O dataset inicial possui 85.485 amostras e 27 colunas. Após a limpeza, nenhuma linha foi removida, indicando que não foram encontrados valores ausentes em colunas obrigatórias nem registros duplicados.

Após a separação entre treino e teste, os dados ficaram com 25 features de entrada, correspondentes às características espectrais dos sinais EEG.

In [ ]:
print("X_train:", result.X_train.shape)
print("X_test:", result.X_test.shape)
print("y_train:", result.y_train.shape)
print("y_test:", result.y_test.shape)
print("Features:", len(result.feature_columns))

In [ ]:
import pandas as pd

print("Treino:")
print(pd.Series(result.y_train).value_counts().sort_index())

print("\nTeste:")
print(pd.Series(result.y_test).value_counts().sort_index())

In [ ]:
print("Média do treino:", result.X_train.mean().round(4))
print("Desvio padrão do treino:", result.X_train.std().round(4))

A média próxima de zero e o desvio padrão próximo de um indicam que a padronização foi aplicada corretamente ao conjunto de treino. Essa etapa é importante para modelos sensíveis à escala das variáveis, como redes neurais, SVM e regressão logística.

In [ ]:
train_percent = pd.Series(result.y_train).value_counts(normalize=True).sort_index() * 100
test_percent = pd.Series(result.y_test).value_counts(normalize=True).sort_index() * 100

print("Percentual no treino:")
print(train_percent.round(2))

print("\nPercentual no teste:")
print(test_percent.round(2))

In [ ]:
from pathlib import Path
import numpy as np

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

np.save(PROCESSED_DIR / "X_train_base.npy", result.X_train)
np.save(PROCESSED_DIR / "X_test_base.npy", result.X_test)
np.save(PROCESSED_DIR / "y_train.npy", result.y_train)
np.save(PROCESSED_DIR / "y_test.npy", result.y_test)
np.save(PROCESSED_DIR / "users_train.npy", result.users_train)
np.save(PROCESSED_DIR / "users_test.npy", result.users_test)

print("Arquivos processados salvos em data/processed.")